# Quickstart — `causal_ai_av`

End-to-end tour of the DevKit:

1. Load the corpus.
2. Inspect a single clip's annotation.
3. Run a query in the DSL.
4. Read the `MatchSet` back.


In [ ]:
from pathlib import Path
from causal_ai_av.dataset import CausalAVDataset

CORPUS = Path("/home/horde/01_json_annotations")
ds = CausalAVDataset(CORPUS)
print(f"{len(ds)} clips")


## 1. Inspect one clip's annotation

`get_sequence` downloads the clip video lazily by default. We
bypass it here and read the parsed annotation directly via the
local IO module — quick and offline-friendly.


In [ ]:
from causal_ai_av.io import load_dir
bundles = load_dir(CORPUS)
b = bundles[0]
print("clip_id:", b.video.clip_id)
print("schema:", b.schema_version)
print("agents:", len(b.annotation.agents))
print("ego actions:", len(b.annotation.ego_vehicle.actions))
print("environments:", len(b.annotation.environments))


## 2. Run a DSL query against the whole corpus


In [ ]:
ds.count("agent.type = ped")


In [ ]:
ds.count("agent.type = ped and env.type = crosswalk")


## 3. `find()` returns the full `MatchSet`

Every match is `(clip_id, entity, interval)`. The `.clips()` /
`.entities()` / `.intervals()` projections pull each column.


In [ ]:
matches = ds.find("agent.type = ped and env.type = crosswalk")
print(f"{len(matches)} matches across {len(set(matches.clips()))} distinct clips")
list(set(matches.clips()))[:3]


## 4. A per-clip query

The same DSL runs on a single `AnnotationBundle` via
`find_on_bundle`. Returns a `MatchSet` scoped to that clip only.


In [ ]:
from causal_ai_av.query import find_on_bundle
single = find_on_bundle(b, "agent.type = vehicle")
single.entities()[:3]
